# Accessing BigQuery Datasets in Workbench

**Dataset:** `wb-crisp-bean-1269.temporary_data`

This notebook demonstrates how to query a BigQuery dataset from a Verily Workbench
cloud environment using the `google-cloud-bigquery` Python client library.

The dataset contains national wastewater pathogen surveillance data, including
measurements from over 2,000 treatment plants across the United States.

## Overview

BigQuery datasets attached to your Workbench workspace are accessible from notebook
cloud environments using standard Google Cloud client libraries. Authentication is
handled automatically by the environment — no API keys or service account setup is
required.

### Objective

Use this notebook to learn how to:

- Connect to a BigQuery dataset from Python
- List available tables in a dataset
- Inspect table schema and metadata
- Run SQL queries and load results into a pandas DataFrame
- Compute summary statistics with aggregation queries

### Costs

This notebook runs a small number of queries against a ~400 MB dataset. The total
data scanned will be well under 1 GB.

## Setup

Import the BigQuery client library and configure the dataset reference.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

PROJECT = "wb-crisp-bean-1269"
DATASET = "temporary_data"
DATASET_REF = f"{PROJECT}.{DATASET}"

## 1. List Tables in the Dataset

A BigQuery dataset can contain multiple tables. The cell below lists all tables
available in the dataset.

In [ ]:
tables = list(client.list_tables(DATASET_REF))

print(f"Found {len(tables)} table(s) in {DATASET_REF}:\n")
for t in tables:
    print(f"  - {t.table_id} ({t.table_type})")

## 2. Inspect Table Schema

Before querying, it is useful to understand the structure of the data. The cell
below retrieves metadata for the table, including the number of rows, the size
on disk, and the full column schema.

In [ ]:
table = client.get_table(f"{DATASET_REF}.all")

print(f"Table:  {table.full_table_id}")
print(f"Rows:   {table.num_rows:,}")
print(f"Size:   {table.num_bytes / 1e6:.1f} MB")
print(f"\nSchema ({len(table.schema)} columns):")
for field in table.schema:
    print(f"  {field.name:30s} {field.field_type:10s}  {field.description or ''}")

## 3. Preview Data

Run a simple `SELECT *` query with a `LIMIT` clause to preview the first few rows.
The results are returned as a pandas DataFrame for convenient display.

In [ ]:
query = f"""
SELECT *
FROM `{DATASET_REF}.all`
LIMIT 10
"""

df = client.query(query).to_dataframe()
df

## 4. Summary Statistics

Aggregation queries are a good way to get a high-level picture of the dataset.
The query below computes counts, distinct values, and the date range of sample
collections.

In [ ]:
summary_query = f"""
SELECT
  COUNT(*)                       AS total_rows,
  COUNT(DISTINCT plant_name)     AS distinct_plants,
  COUNT(DISTINCT pathogen)       AS distinct_pathogens,
  COUNT(DISTINCT plant_region)   AS distinct_regions,
  MIN(sample_collection_date)    AS earliest_date,
  MAX(sample_collection_date)    AS latest_date
FROM `{DATASET_REF}.all`
"""

summary = client.query(summary_query).to_dataframe()
summary.T

## Next Steps

You now have the building blocks to query any BigQuery dataset in your Workbench
workspace. From here you can:

- Write more complex SQL queries with `WHERE`, `GROUP BY`, and `JOIN` clauses
- Use `pandas-gbq` or `%%bigquery` cell magics as alternative query interfaces
- Save query results to new BigQuery tables or export to Cloud Storage

In [ ]:
print("Done. All queries completed successfully.")